In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
from IPython.display import display

# load & preprocess data
import os
print("Working dir:", os.getcwd())

file_path = '../datasets/compas-scores.csv'
df = pd.read_csv(file_path)

df = df[df['race'].isin(['African-American', 'Caucasian'])]
df = df[df['decile_score'] != 5]
df['predicted'] = df['decile_score'].ge(6).astype(int)
df['true']      = df['two_year_recid']
df['score']     = df['decile_score']

groups = ['African-American', 'Caucasian']
scores = sorted(df['score'].unique())

Working dir: /Users/adityamittal/Desktop/git/ucdavis-projects/differential-privacy-fairness/code


In [14]:
# 1) Confusion matrices as a DataFrame
confusion_rows = []
for grp in groups:
    sub = df[df['race'] == grp]
    tn, fp, fn, tp = confusion_matrix(sub['true'], sub['predicted']).ravel()
    confusion_rows.append({'Race': grp, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})

conf_df = pd.DataFrame(confusion_rows).set_index('Race')
conf_df.index.name = None
display(conf_df.style
        .set_caption("Equalized‑Odds Confusion Matrices")
        .format("{:,.0f}"))

,TN,FP,FN,TP
African-American,990,616,532,"1,193"
Caucasian,"1,139",219,461,394


In [ ]:
# 2) sufficiency scores by group
prob_rows = []
for sc in scores:
    for grp in groups:
        sub = df[(df['race']==grp) & (df['score']==sc)]
        p1 = sub['true'].mean() if len(sub) else np.nan
        p0 = 1 - p1 if not np.isnan(p1) else np.nan
        prob_rows.append({'Score': sc, 'Race': grp, 'P(Y=1)': p1, 'P(Y=0)': p0})

prob_df = pd.DataFrame(prob_rows)
prob_pivot = (
    prob_df
    .pivot(index='Score', columns='Race', values=['P(Y=1)', 'P(Y=0)'])
)

display(prob_pivot.style
        .set_caption("Recidivism Probabilities by Score and Race")
        .format("{:.2f}"))